### Chestradqwen: Training data generation

Set up environment (including `assume --env` credentials if required)

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from datagen import MESARuntimeBatchGenerator, TrainingDataUploader, BatchOutputCorrector
from chestradschema.prompt_builder import PromptBuilder
from chestradschema import Schema

In [ ]:
pb = PromptBuilder()

#### Download documents from batch

In [ ]:
gen = MESARuntimeBatchGenerator(
    system_prompt=pb.build_datagen_prompt(),
    schema=Schema,
    schema_name="chestradschema",
    document_batches=["chestradqwen-batch-2026-11-05-001.tar.gz", "normal-batch-2025-12-31-001.tar.gz"],
    base_url=os.environ["LLM__MESA_RUNTIME__BASE_URL"],
    username=os.environ["LLM__MESA_RUNTIME__USERNAME"],
    password=os.environ["LLM__MESA_RUNTIME__PASSWORD"],
    model_name="qwen3.5",
)

In [ ]:
gen.get_document_files_count()

#### Sanity check

##### Start batch generation

In [ ]:
run_id: int = gen.generate_via_batch(100, os.environ["BUCKET"])

In [ ]:
gen.get_batch_run(run_id)

##### Download and parse batch outputs

In [ ]:
gen.extract_batch_output(os.environ["BUCKET"])

#### Start datagen

In [ ]:
run_id: int = gen.generate_via_batch(4000, os.environ["BUCKET"])

In [ ]:
gen.get_batch_run(run_id)

In [ ]:
gen.extract_batch_output(os.environ["BUCKET"])

#### Correct batch outputs

In [ ]:
corrector = BatchOutputCorrector(Schema, Path("anthropic_batch_job.jsonl.out"))

In [ ]:
corrector.correct_with_claude(Path("prompts/correction.txt"), pb.build_datagen_prompt(), Path("rules/correction.yaml"))

In [ ]:
corrector.correct_with_rules(Path("rules/correction.yaml"))

#### Upload formatted document:schema pairs as training data

In [ ]:
s3_uri = TrainingDataUploader.upload(
    schema=Schema,
    schema_name="chestradschema",
    system_prompt=pb.build_main_prompt(),
    short_description="chestradqwen-batch",
    long_description="Training data for Chestradqwen",
    input_folder=Path("./data/trainingdata"),
)